# Imports and initial inputs

In [ ]:

import os, time, json
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree, connected_components

# Supply as string or list:
# "8 82 7 99.5 4 16 6 1124 921 910 214 0.036057234 1.357350751 0.133223543 1.069541905 11 5"
# Value order (1-indexed):
# 1 p_K_DENSITY
# 2 p_KEEP_PERCENT
# 3 p_K_GRAPH
# 4 p_EDGE_THRESH_VAL
# 5 p_DIST_FACTOR
# 6 p_MIN_COMP_SIZE
# 7 combo_idx  (optional; output naming only)
# remaining summary columns are informational only and
# only the first 6 parameters are required for the pipeline run.
PARAM_ROW = "8 82 7 99.5 4 16 6 1124 921 910 214 0.036057234 1.357350751 0.133223543 1.069541905 11 5"

INPUT_CSV = "6.csv"            
OUTDIR = "results_parsa"       
OUTNAME_PREFIX = "from_params"
N_JOBS = 1
LARGE_COMP_MIN = 30            # optional large_comp_min for run_prune_pipeline







## ----------------- IO helper functions -----------------


In [ ]:
def load_csv_xyz(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Input file not found: {path}")
    df = pd.read_csv(path, comment='#')
    cols = df.columns.tolist()
    if set(['X','Y','Z']).issubset(set(cols)):
        arr = df[['X','Y','Z']].values
    else:
        arr = df.iloc[:, :3].values
    return arr.astype(float), df

def save_csv_xyz(path, pts, header="X,Y,Z"):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    np.savetxt(path, pts, delimiter=",", header=header, comments='')


## ----------------- kNN graph and MST -----------------


In [ ]:

def build_knn_graph(points, k=8, n_jobs=1):
    n = len(points)
    if n == 0:
        return csr_matrix((n,n))
    k_use = min(k+1, n)
    nbrs = NearestNeighbors(n_neighbors=k_use, algorithm='kd_tree', n_jobs=n_jobs).fit(points)
    dists, idxs = nbrs.kneighbors(points)
    rows, cols, data = [], [], []
    for i in range(n):
        for j in range(1, idxs.shape[1]):
            rows.append(i); cols.append(int(idxs[i, j])); data.append(float(dists[i, j]))
    A = csr_matrix((data, (rows, cols)), shape=(n, n))
    A = (A + A.T) / 2.0
    return A

def mst_edges_from_adj(A):
    if A.nnz == 0:
        return []
    mst = minimum_spanning_tree(A)
    mst = mst.tocoo()
    edges = list(zip(mst.row.tolist(), mst.col.tolist(), mst.data.tolist()))
    normalized = []
    for u,v,w in edges:
        u=int(u); v=int(v); w=float(w)
        if u < v:
            normalized.append((u,v,w))
        else:
            normalized.append((v,u,w))
    normalized = list({(u,v,w) for (u,v,w) in normalized})
    return normalized


## ----------------- Cut long edges and remove small components -----------------


In [ ]:

def cut_long_edges_and_components(points, k_graph=12, edge_thresh_mode='percentile', edge_thresh_val=95.0,
                                  dist_factor=6.0, min_comp_size=10, large_comp_min=30, n_jobs=1):
    info = {}
    n = len(points)
    if n == 0:
        return np.zeros(0, dtype=bool), info

    A = build_knn_graph(points, k=k_graph, n_jobs=n_jobs)
    edges = mst_edges_from_adj(A)
    if len(edges) == 0:
        return np.ones(n, dtype=bool), info
    weights = np.array([w for (_,_,w) in edges], dtype=float)
    info['mst_edge_count'] = int(len(weights))
    if edge_thresh_mode == 'percentile':
        thresh = float(np.percentile(weights, edge_thresh_val))
    elif edge_thresh_mode == 'median_std':
        thresh = float(np.median(weights) + edge_thresh_val * np.std(weights))
    else:
        thresh = float(edge_thresh_val)
    info['edge_thresh'] = thresh

    rows, cols, data = [], [], []
    for (u,v,w) in edges:
        if w <= thresh:
            rows.append(u); cols.append(v); data.append(w)
            rows.append(v); cols.append(u); data.append(w)
    if len(rows) == 0:
        labels = np.arange(n)
        sizes = np.bincount(labels, minlength=n)
        info['pruned_edge_count'] = 0
    else:
        B = csr_matrix((data, (rows, cols)), shape=(n, n))
        n_comp, labels = connected_components(B, directed=False, connection='weak')
        sizes = np.bincount(labels)
        info['pruned_edge_count'] = int(len(data)//2)
        info['n_components'] = int(n_comp)

    centroids = np.zeros((len(sizes), 3))
    for i in range(len(sizes)):
        idx = np.where(labels == i)[0]
        if len(idx) > 0:
            centroids[i] = points[idx].mean(axis=0)
        else:
            centroids[i] = np.array([np.nan, np.nan, np.nan])
    large_idxs = np.where(sizes >= large_comp_min)[0]
    if len(large_idxs) == 0:
        order = np.argsort(sizes)[::-1]
        large_idxs = order[:2] if len(order)>1 else order[:1]
    info['large_idxs'] = [int(x) for x in large_idxs.tolist()]

    nn = NearestNeighbors(n_neighbors=min(2, max(2, n)), algorithm='kd_tree').fit(points)
    dists, _ = nn.kneighbors(points)
    median_nn = float(np.median(dists[:, -1]))
    info['median_nn'] = median_nn
    dist_thresh = dist_factor * median_nn
    info['dist_thresh'] = dist_thresh

    keep_comp = np.ones(len(sizes), dtype=bool)
    for comp in range(len(sizes)):
        if sizes[comp] >= min_comp_size:
            keep_comp[comp] = True
            continue
        if len(large_idxs)>0:
            dists_to_large = np.linalg.norm(centroids[large_idxs] - centroids[comp], axis=1)
            dmin = float(np.min(dists_to_large))
        else:
            dmin = float('inf')
        if dmin > dist_thresh:
            keep_comp[comp] = False
        else:
            keep_comp[comp] = True
    mask_keep = np.array([keep_comp[lab] for lab in labels], dtype=bool)
    info['component_sizes'] = [int(x) for x in sizes.tolist()]
    info['kept_components'] = int(np.sum(keep_comp))
    info['total_components'] = int(len(sizes))
    return mask_keep, info


## ----------------- Main pipeline function with parameters -----------------


In [ ]:

def run_prune_pipeline(input_csv, outdir, outname,
                       k_density=8, keep_percent=88.0,
                       k_graph=7, edge_thresh_mode='percentile', edge_thresh_val=98.0,
                       dist_factor=4.0, min_comp_size=15, large_comp_min=30, n_jobs=1):
    t0 = time.time()
    pts, df_orig = load_csv_xyz(input_csv)
    n0 = len(pts)
    print(f"[RUN] loaded {n0} pts from {input_csv}")
    # density keep
    k_use = min(k_density+1, max(2, n0))
    nn = NearestNeighbors(n_neighbors=k_use, algorithm='kd_tree', n_jobs=n_jobs).fit(pts)
    dists, _ = nn.kneighbors(pts)
    kth = dists[:, -1]
    kth_thresh = float(np.percentile(kth, keep_percent))
    mask_density = kth <= kth_thresh
    pts_den = pts[mask_density]
    print(f"[RUN] density: kept {len(pts_den)} / {n0}  (keep_percent={keep_percent}, kth-thresh={kth_thresh:.6g})")
    if len(pts_den) == 0:
        print("[ERROR] All points removed by density filter. Relax parameters.")
        stats = {'n_input': n0, 'n_after_density': 0, 'n_final': 0, 'n_removed': n0, 'run_time_s': time.time()-t0,
                 'kth_thresh': kth_thresh}
        return None, None, stats

    mask_keep, info = cut_long_edges_and_components(pts_den,
                                                    k_graph=k_graph,
                                                    edge_thresh_mode=edge_thresh_mode,
                                                    edge_thresh_val=edge_thresh_val,
                                                    dist_factor=dist_factor,
                                                    min_comp_size=min_comp_size,
                                                    large_comp_min=large_comp_min,
                                                    n_jobs=n_jobs)
    pts_final = pts_den[mask_keep]
    t1 = time.time()
    stats = {
        'n_input': n0,
        'n_after_density': int(np.sum(mask_density)),
        'n_final': len(pts_final),
        'n_removed': n0 - len(pts_final),
        'run_time_s': t1 - t0,
        'kth_thresh': kth_thresh,
        'median_nn': info.get('median_nn', None),
        'edge_thresh': info.get('edge_thresh', None),
        'n_components': info.get('n_components', None),
        'kept_components': info.get('kept_components', None),
        'prune_info': info
    }
    os.makedirs(outdir, exist_ok=True)
    outpath = os.path.join(outdir, outname)
    if pts_final is not None:
        save_csv_xyz(outpath, pts_final)
        print("[RUN] saved final ->", outpath)
    return outpath, pts_final, stats


# Wrapper: parse parameters from string/list and run


In [ ]:

def apply_params_direct(param_row, input_csv, outdir, outname_prefix, large_comp_min=30, n_jobs=1):
    # parse string or list
    if isinstance(param_row, str):
        parts = param_row.strip().split()
    elif isinstance(param_row, (list, tuple, np.ndarray)):
        parts = list(map(str, param_row))
    else:
        raise ValueError("param_row must be string or list-like")

    if len(parts) < 6:
        raise ValueError("At least 6 values required: K_DENSITY KEEP_PERCENT K_GRAPH EDGE_THRESH_VAL DIST_FACTOR MIN_COMP_SIZE")

    # map first 6 tokens to parameters
    try:
        p_K_DENSITY = int(float(parts[0]))
        p_KEEP_PERCENT = float(parts[1])
        p_K_GRAPH = int(float(parts[2]))
        p_EDGE_THRESH_VAL = float(parts[3])
        p_DIST_FACTOR = float(parts[4])
        p_MIN_COMP_SIZE = int(float(parts[5]))
    except Exception as e:
        raise ValueError("Cannot parse numeric parameters: " + str(e))

    # optional combo_idx if present
    combo_idx = None
    if len(parts) >= 7:
        try:
            combo_idx = int(float(parts[6]))
        except:
            combo_idx = None

    # build outname
    if combo_idx is not None:
        outname = f"{outname_prefix}_combo{combo_idx}.csv"
    else:
        outname = f"{outname_prefix}_k{p_K_DENSITY}_kp{int(p_KEEP_PERCENT)}_kg{p_K_GRAPH}.csv"

    print("Running pipeline with parameters (from provided row):")
    print(f"  K_DENSITY={p_K_DENSITY}, KEEP_PERCENT={p_KEEP_PERCENT}, K_GRAPH={p_K_GRAPH}, EDGE_THRESH_VAL={p_EDGE_THRESH_VAL},")
    print(f"  DIST_FACTOR={p_DIST_FACTOR}, MIN_COMP_SIZE={p_MIN_COMP_SIZE}, LARGE_COMP_MIN={large_comp_min}")
    outpath, pts_final, stats = run_prune_pipeline(
        input_csv=input_csv, outdir=outdir, outname=outname,
        k_density=p_K_DENSITY, keep_percent=p_KEEP_PERCENT,
        k_graph=p_K_GRAPH, edge_thresh_mode='percentile', edge_thresh_val=p_EDGE_THRESH_VAL,
        dist_factor=p_DIST_FACTOR, min_comp_size=p_MIN_COMP_SIZE, large_comp_min=large_comp_min,
        n_jobs=n_jobs
    )

    # print neat summary
    if stats is not None:
        print("\n=== SUMMARY ===")
        print(f"initial_count: {stats['n_input']}")
        print(f"after_density_count: {stats['n_after_density']}")
        print(f"final_count: {stats['n_final']}")
        print(f"n_removed: {stats['n_removed']}")
        print(f"run_time_s: {stats['run_time_s']:.4f}")
        print(f"kth_thresh: {stats.get('kth_thresh')}")
        print(f"median_nn: {stats.get('median_nn')}")
        print(f"edge_thresh (used): {stats.get('edge_thresh')}")
        print(f"n_components: {stats.get('n_components')}")
        print(f"kept_components: {stats.get('kept_components')}")
    else:
        print("[ERR] No stats returned (likely all removed by density filter).")
    return outpath, pts_final, stats

# ----------------- Direct run with specified PARAM_ROW -----------------
print("Applying parameters from provided row...")
outpath, pts_final, stats = apply_params_direct(PARAM_ROW, INPUT_CSV, OUTDIR, OUTNAME_PREFIX, large_comp_min=LARGE_COMP_MIN, n_jobs=N_JOBS)
print("Done. final saved at:", outpath)
